<a href="https://colab.research.google.com/github/aakilkhanemon/mpox-tri-modal-drug-and-subunit-vaccine/blob/main/Advanced_Lead_Scoring_%26_Binding_Affinity_Evaluation_(GNINA_CNN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import math

try:
    import MDAnalysis as mda
except ImportError:
    !pip install -q mdanalysis
    import MDAnalysis as mda

# Standardized filenames in your sidebar
PDB_FILE = "receptor.pdb"

# Your exact 69-residue list compiled into a clean, space-separated format
TARGET_RESIDUES = (
    "1 33 34 37 67 68 70 76 77 80 81 84 85 94 95 96 97 114 115 116 117 119 "
    "138 139 140 141 154 155 156 157 158 159 161 162 164 165 168 169 186 "
    "187 188 189 200 201 203 204 214 217 218 219 221 223 240 244 245 246 "
    "257 258 259 260 272 273 274 276 285 286 289 292 293"
)

try:
    u = mda.Universe(PDB_FILE)

    # Isolate alpha carbons and backbone anchors strictly on Chain A to center the mass
    selection_string = f"chainID A and resid {TARGET_RESIDUES} and (name CA C N O or backbone)"
    pocket_atoms = u.select_atoms(selection_string)

    # Fail-safe protocol if chain identifiers are stripped
    if len(pocket_atoms) == 0:
        selection_string = f"resid {TARGET_RESIDUES}"
        pocket_atoms = u.select_atoms(selection_string)
        print("[Warning] Falling back to global sequence matching without explicit Chain ID filters.")

    # Compute center of mass and geometric span across the 3D plane
    center = pocket_atoms.center_of_mass()
    positions = pocket_atoms.positions
    coordinate_span = np.max(positions, axis=0) - np.min(positions, axis=0)

    # Add standard structural padding to ensure the ligand can rotate within the box
    suggested_box_dim = np.max(coordinate_span) + 8.0
    final_box_size = math.ceil(suggested_box_dim)

    print("\n" + "="*60)
    print(" DYNAMIC BIOPHYSICAL MAPPING COMPLETE")
    print("="*60)
    print(f"Successfully tracked {len(pocket_atoms.residues)} active residues across the target structure.")
    print(f"Calculated Center : X = {center[0]:.3f} | Y = {center[1]:.3f} | Z = {center[2]:.3f}")
    print(f"Computed Box Size : {final_box_size} Ångströms")
    print("-"*60)
    print("RUN THIS UNIX COMMAND IN YOUR NEXT CELL TO EXECUTE DOCKING:")
    print(f"!./gnina -r {PDB_FILE} -l ligand.sdf \\")
    print(f"        --center_x {center[0]:.3f} --center_y {center[1]:.3f} --center_z {center[2]:.3f} \\")
    print(f"        --size_x {final_box_size} --size_y {final_box_size} --size_z {final_box_size} \\")
    print(f"        --num_modes 5 --device 0 --cnn_scoring rescore --out multi_residue_docked.sdf")
    print("="*60 + "\n")

except Exception as e:
    print(f"❌ Pipeline Broken: {e}")
    print("Verify that 'receptor.pdb' is completely uploaded and contains these exact residue coordinates.")


 DYNAMIC BIOPHYSICAL MAPPING COMPLETE
Successfully tracked 69 active residues across the target structure.
Calculated Center : X = -12.818 | Y = 13.078 | Z = -32.588
Computed Box Size : 61 Ångströms
------------------------------------------------------------
RUN THIS UNIX COMMAND IN YOUR NEXT CELL TO EXECUTE DOCKING:
!./gnina -r receptor.pdb -l ligand.sdf \
        --center_x -12.818 --center_y 13.078 --center_z -32.588 \
        --size_x 61 --size_y 61 --size_z 61 \
        --num_modes 5 --device 0 --cnn_scoring rescore --out multi_residue_docked.sdf



In [ ]:
import os
import pandas as pd
from rdkit import Chem

output_file = 'multi_residue_docked.sdf'

if not os.path.exists(output_file):
    print(f"❌ Error: '{output_file}' not found. Verify the terminal execution log.")
else:
    supplier = Chem.SDMolSupplier(output_file)
    compiled_poses = []

    for i, mol in enumerate(supplier):
        if mol is None: continue
        compiled_poses.append({
            "Pose_Mode": i + 1,
            "Vina_Energy_kcal_mol": float(mol.GetProp('minimizedAffinity')),
            "CNN_Pose_Confidence": float(mol.GetProp('CNNscore')),
            "CNN_Predicted_pKd": float(mol.GetProp('CNNaffinity'))
        })

    df = pd.DataFrame(compiled_poses)
    print("==========================================================")
    print(" ✅ DEEP LEARNING LEADERBOARD GENERATED")
    print("==========================================================")
    print(df.to_string(index=False))

ModuleNotFoundError: No module named 'rdkit'

In [ ]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out multi_residue_docked.sdf

/bin/bash: line 1: ./gnina: No such file or directory


In [ ]:
# 1. Install required python libraries modularly and quietly
!pip install -q rdkit mdanalysis pandas

# 2. Re-download and activate the compiled GNINA binary executable
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

print("==========================================================")
print("✅ WORKSPACE INITIALIZATION COMPONENT-BY-COMPONENT COMPLETE")
print("==========================================================")
print("• RDKit Chemoinformatics Engine: Active")
print("• MDAnalysis Structural Suite  : Active")
print("• GNINA Deep Learning Driver   : Compiled & Ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 57.4 MB/s eta 0:00:00
✅ WORKSPACE INITIALIZATION COMPONENT-BY-COMPONENT COMPLETE
• RDKit Chemoinformatics Engine: Active
• MDAnalysis Structural Suite  : Active
• GNINA Deep Learning Driver   : Compiled & Ready


In [ ]:
!./gnina -r receptor.pdb -l ligand.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.1 master:e4cb380+   Built Dec 18 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina -r receptor.pdb -l ligand.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders

Using random seed: -120797870

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
3328725 | po

In [ ]:
!./gnina -r receptor.pdb -l ligand2.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.1 master:e4cb380+   Built Dec 18 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina -r receptor.pdb -l ligand2.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders

Using random seed: 513974082

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
25101755 | p

In [ ]:
!./gnina -r receptor.pdb -l ligand3.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.1 master:e4cb380+   Built Dec 18 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina -r receptor.pdb -l ligand3.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders

Using random seed: -1450013966

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
11485687 |

In [ ]:
!./gnina -r receptor.pdb -l control.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.1 master:e4cb380+   Built Dec 18 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina -r receptor.pdb -l control.sdf --center_x -12.818 --center_y 13.078 --center_z -32.588 --size_x 61 --size_y 61 --size_z 61 --num_modes 5 --device 0 --cnn_scoring rescore --out massive_pocket_docked.sdf
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders

Using random seed: -1133809408

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
16124688 |